# Bank Account Fraud Detection using Classical Machine Learning

This notebook demonstrates fraud detection using classical ML algorithms on the **Bank Account Fraud Dataset (NeurIPS 2022)**.

## Dataset
- **Source**: [Kaggle - Bank Account Fraud Dataset](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022)
- Real-world bank account fraud detection data
- Highly imbalanced dataset (typical for fraud scenarios)

## Reference
This implementation follows best practices from the [**Fraud Detection Handbook**](https://fraud-detection-handbook.github.io/fraud-detection-handbook/), particularly:
- **Chapter 4**: Baseline Fraud Detection System
- **Chapter 5**: Validation Strategies
- **Chapter 6**: Model Selection
- **Chapter 7**: Feature Engineering

## Algorithms Implemented
1. Logistic Regression
2. Decision Trees
3. Random Forest
4. Support Vector Machines (SVM)
5. K-Nearest Neighbors (KNN)
6. Naive Bayes
7. Gradient Boosting

## Key Fraud Detection Metrics
- Precision, Recall, F1-Score
- **Card Precision@k** (top-k predictions)
- ROC-AUC and PR-AUC
- Average Precision Score

## 1. Import Required Libraries

In [1]:
# Data manipulation and analysis
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Preprocessing and Evaluation
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                            accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, roc_curve, 
                            precision_recall_curve, average_precision_score,
                            auc)

# For imbalanced data handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

print("✓ All libraries imported successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")

ModuleNotFoundError: No module named 'imblearn'

## 2. Load Bank Account Fraud Dataset

**Dataset Information:**
- **Source**: NeurIPS 2022 Bank Account Fraud Dataset
- **Task**: Binary classification (fraud vs legitimate accounts)
- **Challenge**: Highly imbalanced dataset typical in fraud detection

**Note**: Download the dataset from [Kaggle](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022) and place `Base.csv` in the current directory.

In [ ]:
# Load the Bank Account Fraud Dataset
# Make sure you've downloaded Base.csv from Kaggle and placed it in the current directory

try:
    # Try to load the dataset
    df = pd.read_csv('Base.csv')
    print("✓ Dataset loaded successfully from Base.csv")
except FileNotFoundError:
    print("⚠️  Base.csv not found. Creating a synthetic dataset for demonstration...")
    print("    Please download from: https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022")
    print()
    
    # Create synthetic data similar to the Bank Account Fraud dataset structure
    np.random.seed(42)
    n_samples = 100000
    
    # Generate synthetic features similar to bank account fraud dataset
    df = pd.DataFrame({
        'income': np.random.exponential(50000, n_samples),
        'name_email_similarity': np.random.uniform(0, 1, n_samples),
        'prev_address_months_count': np.random.poisson(24, n_samples),
        'current_address_months_count': np.random.poisson(18, n_samples),
        'customer_age': np.random.normal(40, 15, n_samples).clip(18, 90),
        'days_since_request': np.random.uniform(0, 30, n_samples),
        'intended_balcon_amount': np.random.exponential(5000, n_samples),
        'zip_count_4w': np.random.poisson(2, n_samples),
        'velocity_6h': np.random.poisson(1, n_samples),
        'velocity_24h': np.random.poisson(2, n_samples),
        'velocity_4w': np.random.poisson(5, n_samples),
        'bank_branch_count_8w': np.random.poisson(3, n_samples),
        'date_of_birth_distinct_emails_4w': np.random.poisson(1, n_samples),
        'credit_risk_score': np.random.normal(500, 100, n_samples).clip(300, 850),
        'email_is_free': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
        'phone_home_valid': np.random.choice([0, 1], n_samples, p=[0.2, 0.8]),
        'phone_mobile_valid': np.random.choice([0, 1], n_samples, p=[0.15, 0.85]),
        'bank_months_count': np.random.poisson(36, n_samples),
        'has_other_cards': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
        'proposed_credit_limit': np.random.exponential(3000, n_samples),
        'fraud_bool': np.random.choice([0, 1], n_samples, p=[0.94, 0.06])  # 6% fraud rate
    })
    
    print("✓ Synthetic dataset created for demonstration")

print(f"\n{'='*70}")
print(f"📊 DATASET OVERVIEW")
print(f"{'='*70}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n🎯 Target Variable: {'fraud_bool' if 'fraud_bool' in df.columns else 'N/A'}")

# Display fraud distribution
if 'fraud_bool' in df.columns:
    fraud_col = 'fraud_bool'
elif 'is_fraud' in df.columns:
    fraud_col = 'is_fraud'
else:
    # Try to identify the target column
    fraud_col = df.columns[-1]

print(f"\n📈 Class Distribution:")
print(df[fraud_col].value_counts())
fraud_rate = (df[fraud_col].sum() / len(df)) * 100
print(f"\n⚠️  Fraud Rate: {fraud_rate:.2f}% (Highly Imbalanced!)")
print(f"{'='*70}")

## 3. Exploratory Data Analysis

In [ ]:
# Display first few rows
print("📋 First 5 rows of the dataset:")
print(df.head())
print("\n" + "="*80 + "\n")

# Basic statistics
print("📊 Dataset Statistics:")
print(df.describe())
print("\n" + "="*80 + "\n")

# Check for missing values
print("🔍 Missing Values Check:")
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    print(missing_counts[missing_counts > 0])
    print(f"\nTotal missing values: {missing_counts.sum()}")
else:
    print("✓ No missing values found!")
print("\n" + "="*80 + "\n")

# Data types
print("📝 Data Types:")
print(df.dtypes)
print("\n" + "="*80 + "\n")

# Feature information
print("📌 Dataset Columns:")
print(f"Total features: {len(df.columns)}")
print(f"Numeric features: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"Categorical features: {df.select_dtypes(exclude=[np.number]).shape[1]}")

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot with log scale for better visualization of imbalance
counts = df[fraud_col].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[0].set_title('Class Distribution (⚠️ Highly Imbalanced!)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_yscale('log')  # Log scale to visualize imbalance better
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
colors = ['#2ecc71', '#e74c3c']
df[fraud_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.2f%%', 
                                   colors=colors, labels=['Legitimate', 'Fraud'],
                                   startangle=90, explode=[0, 0.1])
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f"\n💡 Key Insight from Fraud Detection Handbook:")
print(f"   This extreme imbalance ({fraud_rate:.2f}% fraud) is typical in fraud detection.")
print(f"   Traditional accuracy is NOT a good metric - focus on Precision, Recall, and AUC.")

In [ ]:
# Select numeric features only for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove target column from correlation analysis
numeric_features = [col for col in numeric_cols if col not in [fraud_col]]

# Correlation heatmap (for first 15 numeric features)
n_features_to_show = min(15, len(numeric_features))
plt.figure(figsize=(14, 12))
correlation_matrix = df[numeric_features[:n_features_to_show]].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title(f'Feature Correlation Heatmap (First {n_features_to_show} Features)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📖 Reference: Fraud Detection Handbook - Chapter 7")
print(f"   Feature engineering and understanding feature correlations")
print(f"   helps identify redundant features and potential issues.")

## 4. Data Preprocessing and Splitting

In [ ]:
# Separate features and target
X = df.drop(fraud_col, axis=1)
y = df[fraud_col]

# Handle categorical variables if present
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if len(categorical_cols) > 0:
    print(f"Encoding {len(categorical_cols)} categorical features...")
    le = LabelEncoder()
    for col in categorical_cols:
        X[col] = le.fit_transform(X[col].astype(str))

# Split data using stratified split (important for imbalanced data)
# As per Fraud Detection Handbook Chapter 5: Proper validation strategy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"{'='*70}")
print(f"📂 DATA SPLITTING (Stratified)")
print(f"{'='*70}")
print(f"Training set size: {X_train.shape[0]:,} samples")
print(f"Testing set size: {X_test.shape[0]:,} samples")
print(f"\n🎯 Fraud Distribution:")
print(f"Training set fraud: {y_train.sum():,} ({(y_train.sum()/len(y_train))*100:.2f}%)")
print(f"Testing set fraud: {y_test.sum():,} ({(y_test.sum()/len(y_test))*100:.2f}%)")
print(f"\n💡 Using stratified split to maintain fraud ratio in both sets")
print(f"{'='*70}")

In [ ]:
# Feature scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Original feature mean (first feature): {X_train.iloc[:, 0].mean():.4f}")
print(f"Scaled feature mean (first feature): {X_train_scaled[:, 0].mean():.4f}")
print(f"Scaled feature std (first feature): {X_train_scaled[:, 0].std():.4f}")

## 5. Handling Imbalanced Data with SMOTE

**From Fraud Detection Handbook (Chapter 4):**
- Imbalanced data is the norm in fraud detection
- Sampling strategies: SMOTE (over-sampling), Under-sampling, or combination
- SMOTE creates synthetic examples of the minority class
- Important: Only apply to training data, never to test data!

In [ ]:
# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:")
print(f"Training samples: {len(y_train)}")
print(f"Fraud cases: {y_train.sum()}")
print(f"Legitimate cases: {len(y_train) - y_train.sum()}")
print(f"\nAfter SMOTE:")
print(f"Training samples: {len(y_train_balanced)}")
print(f"Fraud cases: {y_train_balanced.sum()}")
print(f"Legitimate cases: {len(y_train_balanced) - y_train_balanced.sum()}")

# Visualize the effect of SMOTE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.Series(y_train).value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Before SMOTE', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

pd.Series(y_train_balanced).value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('After SMOTE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

plt.tight_layout()
plt.show()

## 6. Fraud-Specific Evaluation Metrics

**Key Metrics from Fraud Detection Handbook:**
1. **Precision@k (Card Precision)**: Precision in top-k% predictions - critical for operational efficiency
2. **Average Precision**: Summary of precision-recall curve
3. **ROC-AUC**: Overall model discrimination ability
4. **Recall**: Ability to catch fraud cases

In [ ]:
def card_precision_at_k(y_true, y_pred_proba, k=100):
    """
    Card Precision@k: Precision when reviewing top-k transactions
    Critical metric in fraud detection for operational efficiency
    
    Reference: Fraud Detection Handbook - Chapter 6
    """
    # Sort by predicted probability (descending)
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    
    # Get top-k predictions
    top_k_indices = sorted_indices[:k]
    
    # Calculate precision in top-k
    n_fraud_in_top_k = y_true.iloc[top_k_indices].sum() if hasattr(y_true, 'iloc') else y_true[top_k_indices].sum()
    precision_at_k = n_fraud_in_top_k / k
    
    return precision_at_k

def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive fraud detection model evaluation
    Based on Fraud Detection Handbook best practices
    """
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred
    
    # Calculate standard metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    avg_precision = average_precision_score(y_test, y_pred_proba)
    
    # Calculate Card Precision@k for different k values
    k_values = [100, 500, 1000]
    card_precision_scores = {}
    for k in k_values:
        if len(y_test) >= k:
            cp_k = card_precision_at_k(y_test, y_pred_proba, k)
            card_precision_scores[k] = cp_k
    
    # Print metrics
    print(f"\n{'='*70}")
    print(f"🎯 {model_name} - Performance Metrics")
    print(f"{'='*70}")
    print(f"Accuracy:           {accuracy:.4f} (⚠️  Not reliable for imbalanced data)")
    print(f"Precision:          {precision:.4f}")
    print(f"Recall:             {recall:.4f}")
    print(f"F1-Score:           {f1:.4f}")
    print(f"ROC-AUC:            {roc_auc:.4f}")
    print(f"Average Precision:  {avg_precision:.4f}")
    
    if card_precision_scores:
        print(f"\n📊 Card Precision@k (Fraud Detection Handbook metric):")
        for k, cp in card_precision_scores.items():
            print(f"   Precision@{k}:  {cp:.4f} ({cp*100:.2f}% of top-{k} are fraud)")
    print(f"{'='*70}\n")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=False)
    axes[0, 0].set_title(f'{model_name} - Confusion Matrix', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Predicted', fontsize=10)
    axes[0, 0].set_ylabel('Actual', fontsize=10)
    axes[0, 0].set_xticklabels(['Legitimate', 'Fraud'])
    axes[0, 0].set_yticklabels(['Legitimate', 'Fraud'])
    
    # Add text annotations
    axes[0, 0].text(0.5, 1.15, f'TN={tn:,}  FP={fp:,}\nFN={fn:,}  TP={tp:,}', 
                    transform=axes[0, 0].transAxes, ha='center', fontsize=9)
    
    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    axes[0, 1].plot(fpr, tpr, label=f'ROC (AUC={roc_auc:.4f})', linewidth=2.5, color='#e74c3c')
    axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1.5, alpha=0.5)
    axes[0, 1].set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
    axes[0, 1].set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
    axes[0, 1].set_title(f'{model_name} - ROC Curve', fontsize=12, fontweight='bold')
    axes[0, 1].legend(loc='lower right', fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Precision-Recall Curve (more informative for imbalanced data)
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_proba)
    axes[1, 0].plot(recall_curve, precision_curve, linewidth=2.5, 
                    label=f'PR (AP={avg_precision:.4f})', color='#3498db')
    axes[1, 0].axhline(y=y_test.sum()/len(y_test), color='k', linestyle='--', 
                       label='Baseline (No Skill)', linewidth=1.5, alpha=0.5)
    axes[1, 0].set_xlabel('Recall', fontsize=10, fontweight='bold')
    axes[1, 0].set_ylabel('Precision', fontsize=10, fontweight='bold')
    axes[1, 0].set_title(f'{model_name} - Precision-Recall Curve', fontsize=12, fontweight='bold')
    axes[1, 0].legend(loc='upper right', fontsize=10)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Card Precision@k visualization
    if card_precision_scores:
        k_list = list(card_precision_scores.keys())
        cp_list = list(card_precision_scores.values())
        axes[1, 1].bar(range(len(k_list)), cp_list, color='#9b59b6', alpha=0.7)
        axes[1, 1].set_xticks(range(len(k_list)))
        axes[1, 1].set_xticklabels([f'Top {k}' for k in k_list])
        axes[1, 1].set_ylabel('Precision', fontsize=10, fontweight='bold')
        axes[1, 1].set_title('Card Precision@k (Top-k Predictions)', fontsize=12, fontweight='bold')
        axes[1, 1].grid(axis='y', alpha=0.3)
        for i, v in enumerate(cp_list):
            axes[1, 1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    else:
        axes[1, 1].text(0.5, 0.5, 'Insufficient data for\nCard Precision@k', 
                       ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].set_title('Card Precision@k', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Classification report
    print("📋 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'], zero_division=0))
    
    # Return metrics for comparison
    result = {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Avg Precision': avg_precision
    }
    
    # Add Card Precision@k scores
    for k, cp in card_precision_scores.items():
        result[f'CP@{k}'] = cp
    
    return result

print("✓ Fraud detection evaluation functions created successfully!")
print("📖 Implementing Card Precision@k from Fraud Detection Handbook")

In [ ]:
# Train Logistic Regression model
print("Training Logistic Regression model...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
lr_results = evaluate_model(lr_model, X_test_scaled, y_test, "Logistic Regression")

## 8. Model 2: Decision Tree Classifier

In [ ]:
# Train Decision Tree model
print("Training Decision Tree model...")
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=20)
dt_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
dt_results = evaluate_model(dt_model, X_test_scaled, y_test, "Decision Tree")

## 9. Model 3: Random Forest Classifier

In [ ]:
# Train Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
rf_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
rf_results = evaluate_model(rf_model, X_test_scaled, y_test, "Random Forest")

In [ ]:
# Feature Importance from Random Forest
# Reference: Fraud Detection Handbook Chapter 7 - Feature Engineering
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 20 features
top_n = min(20, len(feature_importance))
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(top_n)
colors = plt.cm.viridis(np.linspace(0, 1, top_n))
plt.barh(range(top_n), top_features['Importance'].values, color=colors, alpha=0.8)
plt.yticks(range(top_n), top_features['Feature'].values)
plt.xlabel('Importance Score', fontsize=12, fontweight='bold')
plt.ylabel('Features', fontsize=12, fontweight='bold')
plt.title(f'Random Forest - Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Top 10 Most Important Features for Fraud Detection:")
print(f"{'='*70}")
for idx, row in feature_importance.head(10).iterrows():
    print(f"  {row['Feature']:.<50} {row['Importance']:.4f}")
print(f"{'='*70}")
print(f"\n💡 Fraud Detection Handbook Chapter 7:")
print(f"   Feature importance helps understand what drives fraud predictions")
print(f"   and can guide feature engineering efforts.")

## 10. Model 4: Support Vector Machine (SVM)

In [ ]:
# Train SVM model (using a subset for faster training)
print("Training Support Vector Machine model...")
# Using a subset for faster training (SVM can be slow on large datasets)
subset_size = 5000
indices = np.random.choice(len(X_train_balanced), subset_size, replace=False)
X_train_subset = X_train_balanced[indices]
y_train_subset = y_train_balanced[indices]

svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_subset, y_train_subset)
print("Training completed!")

# Evaluate the model
svm_results = evaluate_model(svm_model, X_test_scaled, y_test, "Support Vector Machine")

## 11. Model 5: K-Nearest Neighbors (KNN)

In [ ]:
# Train KNN model
print("Training K-Nearest Neighbors model...")
knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
knn_results = evaluate_model(knn_model, X_test_scaled, y_test, "K-Nearest Neighbors")

## 12. Model 6: Naive Bayes

In [ ]:
# Train Naive Bayes model
print("Training Gaussian Naive Bayes model...")
nb_model = GaussianNB()
nb_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
nb_results = evaluate_model(nb_model, X_test_scaled, y_test, "Gaussian Naive Bayes")

## 13. Model 7: Gradient Boosting

In [ ]:
# Train Gradient Boosting model
print("Training Gradient Boosting model...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, 
                                      max_depth=5, random_state=42)
gb_model.fit(X_train_balanced, y_train_balanced)
print("Training completed!")

# Evaluate the model
gb_results = evaluate_model(gb_model, X_test_scaled, y_test, "Gradient Boosting")

## 14. Model Comparison

In [ ]:
# Compile all results into a comparison DataFrame
results_df = pd.DataFrame([
    lr_results,
    dt_results,
    rf_results,
    svm_results,
    knn_results,
    nb_results,
    gb_results
])

# Display the comparison table
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Sort by F1-Score
results_df_sorted = results_df.sort_values('F1-Score', ascending=False)
print("\n\nModels ranked by F1-Score:")
print(results_df_sorted[['Model', 'F1-Score']].to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = plt.cm.viridis(np.linspace(0, 1, len(results_df)))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    sorted_df = results_df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=colors)
    
    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.4f}', 
                ha='left', va='center', fontsize=9, fontweight='bold')
    
    ax.set_xlabel(metric, fontsize=11, fontweight='bold')
    ax.set_ylabel('Model', fontsize=11, fontweight='bold')
    ax.set_title(f'Model Comparison - {metric}', fontsize=12, fontweight='bold')
    ax.set_xlim([0, 1.0])
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves comparison for all models
plt.figure(figsize=(12, 8))

models_dict = {
    'Logistic Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'SVM': svm_model,
    'KNN': knn_model,
    'Naive Bayes': nb_model,
    'Gradient Boosting': gb_model
}

colors = ['blue', 'green', 'red', 'purple', 'orange', 'brown', 'pink']

for (name, model), color in zip(models_dict.items(), colors):
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})', 
             linewidth=2, color=color)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curves - All Models Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Key Insights and Recommendations

In [ ]:
# Identify the best performing model based on different metrics
best_by_f1 = results_df_sorted.iloc[0]
best_by_recall = results_df.sort_values('Recall', ascending=False).iloc[0]
best_by_roc = results_df.sort_values('ROC-AUC', ascending=False).iloc[0]

print("\n" + "="*80)
print("🎯 KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)

print(f"\n1. BEST PERFORMING MODELS BY METRIC:")
print(f"   {'─'*76}")
print(f"   Best by F1-Score:    {best_by_f1['Model']:<25} (F1: {best_by_f1['F1-Score']:.4f})")
print(f"   Best by Recall:      {best_by_recall['Model']:<25} (Recall: {best_by_recall['Recall']:.4f})")
print(f"   Best by ROC-AUC:     {best_by_roc['Model']:<25} (AUC: {best_by_roc['ROC-AUC']:.4f})")

print(f"\n2. MODEL RANKINGS BY F1-SCORE:")
print(f"   {'─'*76}")
for i, row in enumerate(results_df_sorted.itertuples(), 1):
    model_name = row.Model
    f1_val = row._5  # F1-Score column
    print(f"   {i}. {model_name:<35} F1: {f1_val:.4f}")

print(f"\n3. FRAUD DETECTION HANDBOOK INSIGHTS:")
print(f"   {'─'*76}")
print(f"   ✓ Ensemble methods typically outperform single models")
print(f"   ✓ SMOTE helps balance training data (Chapter 4)")
print(f"   ✓ Card Precision@k is crucial for operational efficiency")
print(f"   ✓ PR-AUC more informative than ROC-AUC for imbalanced data")
print(f"   ✓ Recall critical to minimize missed fraud (false negatives)")
print(f"   ✓ Precision reduces investigation costs (false positives)")

print(f"\n4. PRODUCTION DEPLOYMENT RECOMMENDATIONS:")
print(f"   {'─'*76}")
print(f"   📌 Use ensemble models (Random Forest/Gradient Boosting)")
print(f"   📌 Implement threshold tuning based on business costs")
print(f"   📌 Monitor model performance over time (concept drift)")
print(f"   📌 Regular retraining with new fraud patterns")
print(f"   📌 A/B testing for model updates (Chapter 5)")
print(f"   📌 Consider cost-sensitive learning for asymmetric costs")
print(f"   📌 Implement real-time scoring infrastructure")
print(f"   📌 Maintain model explainability for compliance")

print(f"\n5. NEXT STEPS:")
print(f"   {'─'*76}")
print(f"   🔄 Advanced algorithms: XGBoost, LightGBM, CatBoost")
print(f"   🔄 Hyperparameter tuning with cross-validation")
print(f"   🔄 Feature engineering (Chapter 7)")
print(f"   🔄 Time-based validation splits (Chapter 5)")
print(f"   🔄 Ensemble stacking and blending")
print(f"   🔄 Deep learning approaches (Chapter 8)")

print("\n" + "="*80)

## 16. Threshold Tuning Example (Best Model)

For fraud detection, we can adjust the classification threshold to optimize for recall (catching more fraud) at the cost of precision.

In [ ]:
# Using the best model (likely Random Forest or Gradient Boosting)
# Let's demonstrate threshold tuning with Random Forest
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

print("Threshold Tuning Results:")
print("="*80)

for threshold in thresholds:
    y_pred_threshold = (y_pred_proba_rf >= threshold).astype(int)
    
    acc = accuracy_score(y_test, y_pred_threshold)
    prec = precision_score(y_test, y_pred_threshold)
    rec = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)
    
    threshold_results.append({
        'Threshold': threshold,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"Threshold: {threshold:.1f} | Accuracy: {acc:.4f} | "
          f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

print("="*80)

# Visualize threshold impact
threshold_df = pd.DataFrame(threshold_results)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(threshold_df))
width = 0.2

ax.bar(x - 1.5*width, threshold_df['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x - 0.5*width, threshold_df['Precision'], width, label='Precision', alpha=0.8)
ax.bar(x + 0.5*width, threshold_df['Recall'], width, label='Recall', alpha=0.8)
ax.bar(x + 1.5*width, threshold_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Threshold', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Impact of Classification Threshold on Model Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(threshold_df['Threshold'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 17. Conclusion & References

### 🎓 Summary

This notebook demonstrated **bank account fraud detection** using 7 classical ML algorithms on real-world data:

1. **Logistic Regression** - Linear baseline model
2. **Decision Tree** - Interpretable tree-based classifier
3. **Random Forest** - Ensemble of decision trees
4. **Support Vector Machine (SVM)** - Kernel-based classifier
5. **K-Nearest Neighbors (KNN)** - Instance-based learning
6. **Naive Bayes** - Fast probabilistic classifier
7. **Gradient Boosting** - Sequential ensemble method

---

### 🔑 Key Takeaways from Fraud Detection Handbook

#### Chapter 4: Baseline Fraud Detection System
- ✅ Handle extreme class imbalance with SMOTE/undersampling
- ✅ Use appropriate evaluation metrics (not just accuracy!)
- ✅ Establish baseline performance before complex models

#### Chapter 5: Validation Strategies
- ✅ Stratified splitting maintains fraud distribution
- ✅ Time-aware validation for production scenarios
- ✅ Cross-validation for robust performance estimation

#### Chapter 6: Model Selection & Metrics
- ✅ **Card Precision@k**: Operational efficiency metric
- ✅ **PR-AUC**: Better than ROC-AUC for imbalanced data
- ✅ **Average Precision**: Summarizes precision-recall trade-off
- ✅ Focus on recall to catch fraud, precision to reduce false alarms

#### Chapter 7: Feature Engineering
- ✅ Feature importance reveals fraud indicators
- ✅ Domain knowledge drives effective features
- ✅ Temporal and aggregation features are powerful

---

### 📊 Dataset Information

**Bank Account Fraud Dataset (NeurIPS 2022)**
- **Source**: [Kaggle Dataset](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022)
- Real-world bank account opening fraud detection
- Highly imbalanced (~6% fraud rate typical)
- Features include customer demographics, velocity patterns, credit scores

---

### 📚 References

1. **Fraud Detection Handbook**
   - URL: [https://fraud-detection-handbook.github.io/fraud-detection-handbook/](https://fraud-detection-handbook.github.io/fraud-detection-handbook/)
   - Comprehensive guide to fraud detection best practices
   - Chapters 4-7 especially relevant for baseline systems

2. **Dataset**
   - [Bank Account Fraud Dataset - NeurIPS 2022](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022)
   - Real-world tabular dataset for fraud detection research

3. **Key Papers & Resources**
   - Imbalanced-learn library for SMOTE and sampling techniques
   - Scikit-learn for classical ML implementations
   - Research on Card Precision@k for operational metrics

---

### 🚀 Next Steps & Advanced Topics

1. **Advanced Algorithms**
   - XGBoost, LightGBM, CatBoost for better performance
   - Neural networks and deep learning (Chapter 8)
   - AutoML for automated model selection

2. **Feature Engineering**
   - Temporal aggregations (velocity features)
   - Entity embeddings for categorical variables
   - Domain-specific risk scores

3. **Production Deployment**
   - Real-time scoring pipelines
   - Model monitoring and drift detection
   - A/B testing framework
   - Explainability and compliance (SHAP, LIME)

4. **Advanced Techniques**
   - Cost-sensitive learning
   - Ensemble stacking and blending
   - Graph-based fraud detection
   - Anomaly detection methods

---

### 🎯 Best Practices Checklist

- ✅ Use stratified splits for imbalanced data
- ✅ Never oversample before splitting (data leakage!)
- ✅ Evaluate with multiple metrics (Precision, Recall, F1, AUC)
- ✅ Consider Card Precision@k for operational planning
- ✅ Monitor model performance over time
- ✅ Retrain regularly with new fraud patterns
- ✅ Maintain explainability for compliance
- ✅ Implement threshold tuning based on business costs

In [ ]:
print("="*80)
print("✅ NOTEBOOK COMPLETED SUCCESSFULLY!")
print("="*80)
print("\n🎓 What You've Learned:")
print("  1. ✓ Load and explore the Bank Account Fraud Dataset")
print("  2. ✓ Handle extreme class imbalance with SMOTE")
print("  3. ✓ Implement 7 classical ML algorithms for fraud detection")
print("  4. ✓ Use fraud-specific metrics (Card Precision@k, PR-AUC)")
print("  5. ✓ Compare model performance with proper evaluation")
print("  6. ✓ Apply best practices from Fraud Detection Handbook")
print("  7. ✓ Tune classification thresholds for business objectives")

print("\n📚 Key Resources:")
print("  • Fraud Detection Handbook: https://fraud-detection-handbook.github.io")
print("  • Dataset: https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022")

print("\n🎯 Next Challenge:")
print("  Try implementing advanced algorithms (XGBoost, LightGBM) and")
print("  experiment with feature engineering techniques from Chapter 7!")

print("\n" + "="*80)
print("Happy Fraud Detecting! 🔍🛡️💳")
print("="*80)